# Capitolo 6 — Classificare testo: le parole come vettori (§ 6.8)
Recensioni Yelp tradotte in italiano; media degli embedding contro LSTM.

In [ ]:
import sys; sys.path.insert(0, "..")
from utils import fissa_seme
import dati
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
fissa_seme(42)

import re, collections
from sklearn.metrics import accuracy_score, f1_score, recall_score
tr, va, te = dati.recensioni_yelp()
def binario(d): d = d[d.label != 1].copy(); d["y"] = (d.label == 2).astype(int); return d
tr, va, te = binario(tr), binario(va), binario(te); print(len(tr), len(va), len(te), "positive:", round(tr.y.mean(), 3))

## Tokenizzazione, vocabolario, codifica

In [ ]:
tok = lambda s: re.findall(r"[a-zàèéìòù']+", s.lower())
conteggi = collections.Counter(w for s in tr.translated_text for w in tok(s)); V, L = 5000, 100
vocab = {w: i + 2 for i, (w, _) in enumerate(conteggi.most_common(V - 2))}     # 0 = riempimento, 1 = sconosciuta
print("parole distinte:", len(conteggi), "| coperte dal vocabolario:", round(sum(c for w, c in conteggi.items() if w in vocab) / sum(conteggi.values()), 3))
def codifica(d):
    X = np.zeros((len(d), L), int)
    for i, s in enumerate(d.translated_text):
        ids = [vocab.get(w, 1) for w in tok(s)][:L]; X[i, :len(ids)] = ids
    return torch.from_numpy(X), torch.tensor(d.y.values, dtype=torch.float32)
X_tr, y_tr = codifica(tr); X_va, y_va = codifica(va); X_te, y_te = codifica(te)

## Due modelli

In [ ]:
class MediaEmbedding(nn.Module):
    def __init__(self): super().__init__(); self.emb = nn.EmbeddingBag(V, 64, mode="mean", padding_idx=0); self.fc = nn.Linear(64, 1)
    def forward(self, x): return self.fc(self.emb(x)).squeeze(1)
class LSTMTesto(nn.Module):
    def __init__(self): super().__init__(); self.emb = nn.Embedding(V, 64, padding_idx=0); self.l = nn.LSTM(64, 64, batch_first=True); self.fc = nn.Linear(64, 1)
    def forward(self, x): o, (h, c) = self.l(self.emb(x)); return self.fc(h[-1]).squeeze(1)

perdita_fn = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(float((y_tr == 0).sum() / (y_tr == 1).sum())))
def addestra(M, epoche=30):
    fissa_seme(42); m = M(); opt = torch.optim.Adam(m.parameters(), lr=1e-3, weight_decay=1e-5); migliore = (0, None, 0)
    for epoca in range(epoche):
        m.train(); perm = torch.randperm(len(X_tr))
        for i in range(0, len(X_tr), 32):
            idx = perm[i:i + 32]; opt.zero_grad(); perdita_fn(m(X_tr[idx]), y_tr[idx]).backward(); nn.utils.clip_grad_norm_(m.parameters(), 1.0); opt.step()
        m.eval()
        with torch.no_grad(): pv = (torch.sigmoid(m(X_va)) > 0.5).int().numpy()
        f = f1_score(y_va.numpy(), pv, average="macro")
        if f > migliore[0]: migliore = (f, {k: v.clone() for k, v in m.state_dict().items()}, epoca + 1)
    m.load_state_dict(migliore[1]); m.eval()
    with torch.no_grad(): pt = (torch.sigmoid(m(X_te)) > 0.5).int().numpy()
    print(f"{M.__name__:15s} acc {accuracy_score(y_te.numpy(), pt):.1%}  F1 {f1_score(y_te.numpy(), pt, average='macro'):.3f}  recall negative {recall_score(y_te.numpy(), pt, pos_label=0):.1%}  epoca {migliore[2]}")
    return m
print(f"sempre positiva: acc {max(y_te.mean(), 1 - y_te.mean()):.1%}")
media = addestra(MediaEmbedding); lstm = addestra(LSTMTesto)

## Cosa ha imparato la tabella degli embedding

In [ ]:
E = media.emb.weight.detach(); w = media.fc.weight[0].detach(); punteggio = (E @ w).numpy(); inv = {i: p for p, i in vocab.items()}
ordine = np.argsort(punteggio)
print("negative:", [inv[i] for i in ordine[:15] if i in inv]); print("positive:", [inv[i] for i in ordine[::-1][:15] if i in inv])